In [1]:
%pip install ipykernel
%pip install opencv-python 
%pip install matplotlib
%pip install matplotlib tk
%pip install plotly nbformat
%pip install scipy
%pip install pyqtgraph PyOpenGL PyQt5
%matplotlib qt




Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


... (4 outputs left out)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import threading
import time
from collections import deque
from dataclasses import dataclass

import cv2 as cv
import matplotlib
import numpy as np

'''
def use_window_backend():
    """Notebooks default to an inline backend that closes live figures."""
    try:
        shell = get_ipython()
    except NameError:
        return
    for backend in ("qt", "tk"):
        try:
            shell.run_line_magic("matplotlib", backend)
            return
        except Exception:
            continue


use_window_backend()
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.cm import ScalarMappable  # noqa: E402
from matplotlib.colors import Normalize  # noqa: E402

'''
FRAME_SHAPE = (384, 256, 2)
FRAME_SIZE = int(np.prod(FRAME_SHAPE))
FONT = cv.FONT_HERSHEY_SIMPLEX
@dataclass
class Config:
    camera_index: int = 0
    left_crop: int = 98              # gray (fallback) mode only
    zoom: int = 4
    rows: int = 80                   # time rows kept in the 3D views
    columns: int = 64                # target number of 3D columns across the image
    update_seconds: float = 0.08     # time between 3D updates
    figure_size: tuple = (18, 8)
    blur_x: float = 0.8              # smoothing across image columns (3D views)
    blur_t: float = 1.0              # smoothing across time rows (3D views)

    # temperature scale (raw mode). Starts at temp_min/max, then follows the scene.
    temp_min: float = 10.0
    temp_max: float = 50.0
    auto_scale: bool = True
    scale_low_pct: float = 1.0
    scale_high_pct: float = 99.8
    min_span: float = 6.0            # never zoom the scale tighter than this many degrees
    scale_smoothing: float = 0.08    # 0..1, lower = steadier scale

    # wisps
    ambient_pct: float = 30.0        # "normal" = this percentile of the recent history
    contrast: float = 0.15           # fraction of the scale span that makes a wisp fully strong
    min_strength: float = 0.15       # weaker cells are not drawn


class Scale:
    """Colour/height range shared by the camera view, the bar and both 3D views."""

    def __init__(self, low, high, unit, auto, cfg):
        self.low, self.high, self.unit, self.auto, self.cfg = low, high, unit, auto, cfg
        self.primed = False

    @property
    def span(self):
        return max(self.high - self.low, 1e-6)

    @property
    def label(self):
        return "°C" if self.unit == "C" else self.unit

    def update(self, values):
        """Follow the scene smoothly; rise fast so hot things never clip."""
        if not self.auto:
            return
        cfg = self.cfg
        lo, hi = np.percentile(values[::2, ::2], (cfg.scale_low_pct, cfg.scale_high_pct))
        hi = max(hi, lo + cfg.min_span)
        if not self.primed:
            self.low, self.high, self.primed = float(lo), float(hi), True
            return
        self.low += cfg.scale_smoothing * (lo - self.low)
        self.high += (0.5 if hi > self.high else cfg.scale_smoothing) * (hi - self.high)


def decode_temps(frame):
    """Return a 192x256 array in °C from a raw TC001 frame."""
    data = frame.reshape(FRAME_SHAPE)[FRAME_SHAPE[0] // 2:]
    raw = data[..., 0].astype(np.uint16) + data[..., 1].astype(np.uint16) * 256
    return (raw / 64.0 - 273.15).astype(np.float32)


def decode_gray(frame, crop):
    """Return the cropped brightness image (0 to 255) from a normal colour frame."""
    return cv.cvtColor(frame[:, crop:], cv.COLOR_BGR2GRAY).astype(np.float32)


def open_raw(index):
    """Return a capture that delivers raw TC001 frames, or None if it doesn't."""
    for name, backend in (("DSHOW", cv.CAP_DSHOW), ("MSMF", cv.CAP_MSMF), ("default", cv.CAP_ANY)):
        capture = cv.VideoCapture(index, backend)
        if not capture.isOpened():
            capture.release()
            continue
        capture.set(cv.CAP_PROP_CONVERT_RGB, 0)
        ok, frame = capture.read()
        print(f"[{name}] read ok={ok}, shape={getattr(frame, 'shape', None)}, "
              f"size={getattr(frame, 'size', None)} (need {FRAME_SIZE})")
        if ok and frame.size == FRAME_SIZE:
            return capture
        capture.release()
    return None


class Camera:
    """Reads frames on a background thread so the display never works from stale, queued frames."""

    def __init__(self, cfg):
        self.crop = cfg.left_crop
        self.capture = open_raw(cfg.camera_index)
        self.raw = self.capture is not None
        if not self.raw:
            self.capture = cv.VideoCapture(cfg.camera_index)
        if not self.capture.isOpened():
            raise RuntimeError("Could not open camera. Try another camera_index.")
        if self.raw:
            self.scale = Scale(cfg.temp_min, cfg.temp_max, "C", cfg.auto_scale, cfg)
        else:
            self.scale = Scale(0, 255, "signal", False, cfg)

        self._frame = None
        self._lock = threading.Lock()
        self._ready = threading.Event()
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._grab, daemon=True)
        self._thread.start()

    def _grab(self):
        while not self._stop.is_set():
            ok, frame = self.capture.read()
            with self._lock:
                self._frame = frame if ok else None
            self._ready.set()
            if not ok:
                break

    def read(self):
        """Return the newest temperatures in °C (raw mode) or brightness, or None."""
        if not self._ready.wait(2.0):
            return None
        self._ready.clear()
        with self._lock:
            frame = self._frame
        if frame is None:
            return None
        return decode_temps(frame) if self.raw else decode_gray(frame, self.crop)

    def release(self):
        self._stop.set()
        self._thread.join(timeout=2.0)
        self.capture.release()


class CameraView:
    """Camera window with a hotspot marker and a temperature bar."""

    def __init__(self, cfg, scale):
        self.zoom = cfg.zoom
        self.scale = scale
        self.bar = None

    def make_bar(self, height):
        gradient = np.linspace(255, 0, height).astype(np.uint8)[:, None].repeat(22, axis=1)
        return cv.applyColorMap(gradient, cv.COLORMAP_INFERNO)

    def show(self, values):
        s = self.scale
        scaled = np.clip((values - s.low) / s.span, 0, 1)
        image = cv.applyColorMap(np.uint8(scaled * 255), cv.COLORMAP_INFERNO)
        image = cv.resize(image, None, fx=self.zoom, fy=self.zoom, interpolation=cv.INTER_CUBIC)
        h, w = image.shape[:2]

        y, x = np.unravel_index(np.argmax(values), values.shape)
        cx, cy = int((x + 0.5) * self.zoom), int((y + 0.5) * self.zoom)
        cv.drawMarker(image, (cx, cy), (255, 255, 255), cv.MARKER_CROSS, 14, 1)
        cv.putText(image, f"{values[y, x]:.1f} {s.unit}", (min(cx + 8, w - 120), max(cy - 8, 16)),
                   FONT, 0.5, (255, 255, 255), 1, cv.LINE_AA)

        if self.bar is None or self.bar.shape[0] != h:
            self.bar = self.make_bar(h)
        labels = np.zeros((h, 80, 3), np.uint8)
        for i in range(6):
            frac = i / 5
            row = int(frac * (h - 1))
            cv.line(labels, (0, row), (5, row), (255, 255, 255), 1)
            cv.putText(labels, f"{s.high - frac * s.span:.1f}", (9, min(max(row + 5, 12), h - 3)),
                       FONT, 0.45, (255, 255, 255), 1, cv.LINE_AA)
        spacer = np.zeros((h, 8, 3), np.uint8)
        cv.imshow("TC001", np.hstack([image, spacer, self.bar, labels]))

    @staticmethod
    def quit_requested():
        return cv.waitKey(1) & 0xFF in (ord("q"), 27)


class History:
    """Rolling rows: hottest value in each block of image columns, averaged over each time step."""

    def __init__(self, width, cfg):
        self.step = max(1, width // cfg.columns)
        self.n = width // self.step
        self.x = np.arange(self.n) * self.step + self.step / 2
        self.rows = deque(maxlen=cfg.rows)
        self.total = np.zeros(self.n, np.float32)
        self.count = 0

    def add_frame(self, values):
        blurred = cv.GaussianBlur(values, (0, 0), 1.0)        # suppress single hot pixels
        h = blurred.shape[0]
        blocks = blurred[:, :self.n * self.step].reshape(h, self.n, self.step)
        self.total += blocks.max(axis=(0, 2))                  # hottest pixel per column block
        self.count += 1

    def commit(self):
        if self.count == 0:
            return
        row = self.total / self.count
        if not self.rows:
            self.rows.extend([row] * self.rows.maxlen)
        else:
            self.rows.append(row)
        self.total = np.zeros(self.n, np.float32)
        self.count = 0

    def as_array(self):
        return np.array(self.rows, np.float32)


def style_dark_axes(ax, xlim, ylim, zlabel):
    ax.set_facecolor("black")
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_pane_color((0, 0, 0, 0))
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Image X", color="0.7")
    ax.set_ylabel("Time", color="0.7")
    ax.set_zlabel(zlabel, color="0.7")
    ax.tick_params(colors="0.6")
    ax.view_init(45, -70)


class SurfacePanel:
    """Left panel: surface of the hottest value per column over time, in real units."""

    def __init__(self, ax, history, cfg, scale):
        self.ax, self.scale, self.surf = ax, scale, None
        self.x, self.y = np.meshgrid(history.x, np.arange(cfg.rows))
        style_dark_axes(ax, (0, history.x.max()), (0, cfg.rows - 1), f"Signal ({scale.label})")
        ax.set_title("3D map", color="0.85")

    def draw(self, grid):
        s = self.scale
        if self.surf is not None:
            self.surf.remove()
        self.surf = self.ax.plot_surface(
            self.x, self.y, np.clip(grid, s.low, s.high),
            cmap="inferno", vmin=s.low, vmax=s.high,
            linewidth=0, antialiased=False, shade=False,
            rcount=grid.shape[0], ccount=grid.shape[1],
        )
        self.ax.set_zlim(s.low, s.high)


class WispPanel:
    """Right panel: particles for everything hotter than the scene's normal level."""

    PARTICLES = 2

    def __init__(self, ax, history, cfg, scale):
        self.ax, self.cfg, self.scale, self.history = ax, cfg, scale, history
        self.x, self.y = np.meshgrid(history.x, np.arange(cfg.rows))
        rng = np.random.default_rng(7)
        self.jitter = rng.uniform(-0.5, 0.5, (self.PARTICLES, 3, self.x.size))   # fixed, so no flicker
        self.fade = np.linspace(0.15, 1.0, cfg.rows)[:, None]
        self.ambient = None
        self.top = None
        style_dark_axes(ax, (0, history.x.max()), (0, cfg.rows - 1), f"{scale.label} above normal")
        ax.set_title("Wisps", color="0.85")
        self.cloud = ax.scatter([0], [0], [0], s=1, c=[(0, 0, 0, 0)], depthshade=False, linewidths=0)

    def draw(self, grid):
        s, cfg = self.scale, self.cfg
        target = float(np.percentile(grid, cfg.ambient_pct))
        self.ambient = target if self.ambient is None else self.ambient + 0.05 * (target - self.ambient)

        contrast = max(cfg.contrast * s.span, 1e-6)
        excess = np.clip(grid - self.ambient, 0, None)
        strength = np.clip(excess / contrast, 0, 1) * self.fade
        cells = np.flatnonzero(strength.ravel() > cfg.min_strength)

        target_top = max(4 * contrast, float(excess.max()) * 1.1)
        self.top = target_top if self.top is None else self.top + 0.2 * (target_top - self.top)
        self.ax.set_zlim(0, self.top)

        if not cells.size:
            self.cloud._offsets3d = ([0.0], [0.0], [0.0])
            self.cloud.set_facecolor([(0, 0, 0, 0)])
            self.cloud.set_sizes([1])
            return

        heat = excess.ravel()[cells]
        power = strength.ravel()[cells]
        jit = self.jitter[:, :, cells]                                        # (P, 3, n)
        px = (self.x.ravel()[cells] + jit[:, 0] * self.history.step).ravel()
        py = (self.y.ravel()[cells] + jit[:, 1]).ravel()
        pz = np.clip(heat + jit[:, 2] * contrast * 0.15, 0, None).ravel()
        power = np.tile(power, self.PARTICLES)
        temp = self.ambient + np.tile(heat, self.PARTICLES)                    # absolute value -> same colour bar

        colors = plt.get_cmap("inferno")(np.clip((temp - s.low) / s.span, 0, 1))
        colors[:, 3] = np.clip(power * 1.2, 0, 1)
        self.cloud._offsets3d = (px, py, pz)
        self.cloud.set_facecolor(colors)
        self.cloud.set_sizes(4 + 34 * power)


class SideBySide:
    """One window: surface map (left), wisp map (right), shared temperature bar."""

    def __init__(self, history, cfg, scale):
        plt.ion()
        self.history, self.cfg, self.scale = history, cfg, scale
        self.fig = plt.figure("Thermal 3D + wisps", figsize=cfg.figure_size, facecolor="black")
        left = self.fig.add_subplot(121, projection="3d")
        right = self.fig.add_subplot(122, projection="3d")
        self.fig.subplots_adjust(left=0.01, right=0.9, bottom=0.02, top=0.95, wspace=0.02)

        self.mappable = ScalarMappable(norm=Normalize(scale.low, scale.high), cmap="inferno")
        self.cbar = self.fig.colorbar(self.mappable, cax=self.fig.add_axes([0.93, 0.2, 0.014, 0.6]))
        self.cbar.set_label(f"Temperature ({scale.label})", color="0.8")
        self.cbar.ax.tick_params(colors="0.8")
        self.cbar.outline.set_edgecolor("0.5")

        self.panels = [
            SurfacePanel(left, history, cfg, scale),
            WispPanel(right, history, cfg, scale),
        ]

    def draw(self):
        """Redraw both panels. Returns False once the window is closed."""
        grid = cv.GaussianBlur(self.history.as_array(), (0, 0),
                               sigmaX=self.cfg.blur_x, sigmaY=self.cfg.blur_t)
        self.mappable.set_clim(self.scale.low, self.scale.high)
        self.cbar.update_normal(self.mappable)
        for panel in self.panels:
            panel.draw(grid)
        plt.pause(0.001)
        return plt.fignum_exists(self.fig.number)


def run(cfg=Config()):


    camera = Camera(cfg)
    print("Reading temperatures (°C)." if camera.raw else "Reading brightness only: this camera sends processed frames.")
    view = CameraView(cfg, camera.scale)
    history = maps = None
    last_update = 0.0

    try:
        while True:
            values = camera.read()
            if values is None:
                break
            camera.scale.update(values)

            if history is None:
                history = History(values.shape[1], cfg)
                maps = SideBySide(history, cfg, camera.scale)

            view.show(values)
            history.add_frame(values)

            if time.monotonic() - last_update >= cfg.update_seconds:
                last_update = time.monotonic()
                history.commit()
                if not maps.draw():
                    break

            if view.quit_requested():
                break
    finally:
        camera.release()
        cv.destroyAllWindows()
        plt.close("all")


if __name__ == "__main__":
    run()
    

[DSHOW] read ok=True, shape=(384, 644, 3), size=741888 (need 196608)


: 

In [2]:
#simulated - one window: live map with hotspot boxes, tracked-only view, 3D views and data panels
import threading
import time
from collections import deque
from dataclasses import dataclass
from itertools import takewhile

import cv2 as cv
import matplotlib
import numpy as np


def use_window_backend():
    """Notebooks default to an inline backend that closes live figures."""
    try:
        shell = get_ipython()
    except NameError:
        return
    for backend in ("qt", "tk"):
        try:
            shell.run_line_magic("matplotlib", backend)
            return
        except Exception:
            continue


use_window_backend()
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.patches import Rectangle  # noqa: E402
from matplotlib.widgets import Button  # noqa: E402


FRAME_SHAPE = (384, 256, 2)
FRAME_SIZE = int(np.prod(FRAME_SHAPE))


@dataclass
class Config:
    simulate: bool = True            # True = synthetic scene, False = real TC001
    sim_fps: float = 25.0            # simulated frame rate
    camera_index: int = 0
    left_crop: int = 98              # gray (fallback) mode only
    rows: int = 80                   # time rows kept in the 3D views
    columns: int = 64                # target number of 3D columns across the image
    update_seconds: float = 0.08     # time between live map / data panel updates
    full_update_seconds: float = 0.5 # time between 3D view / chart / data panel updates (slow)
    figure_size: tuple = (18, 10)
    blur_x: float = 0.8              # smoothing across image columns (3D views)
    blur_t: float = 1.0              # smoothing across time rows (3D views)

    # temperature scale (raw mode). Starts at temp_min/max, then follows the scene.
    temp_min: float = 10.0
    temp_max: float = 50.0
    auto_scale: bool = True
    scale_low_pct: float = 1.0
    scale_high_pct: float = 99.8
    min_span: float = 6.0            # never zoom the scale tighter than this many degrees
    scale_smoothing: float = 0.08    # 0..1, lower = steadier scale

    # wisps
    ambient_pct: float = 30.0        # "normal" = this percentile of the recent history
    contrast: float = 0.15           # fraction of the scale span that makes a wisp fully strong
    min_strength: float = 0.15       # weaker cells are not drawn

    # hotspot detection and tracking
    hot_delta: float = 4.0           # a hotspot is at least this much hotter than the scene median...
    hot_fraction: float = 0.12       # ...or this fraction of the colour scale, whichever is larger
    prominence: float = 5.0          # a peak must rise this much above the dip joining it to a
                                     # hotter neighbour to count as its own hotspot
    min_area: int = 20               # pixels; smaller hot blobs are ignored
    min_rise: float = 1.5            # a hotspot's peak must be this far above the hotspot level
    detect_step: int = 2             # detection runs on an image this many times smaller (speed)
    match_distance: float = 30.0     # pixels a hotspot may jump between frames and keep its ID...
    max_speed: float = 80.0          # ...plus this many pixels per second since it was last seen
    confirm_frames: int = 3          # frames before a new hotspot gets an ID and a box
    forget_frames: int = 20          # frames a hotspot may be missing before it is dropped
    focus_margin: int = 4            # pixels of surroundings kept around the picked hotspot
    chart_seconds: float = 30.0      # length of the temperature history chart
    log_length: int = 30000          # samples kept per hotspot (for the chart and CSV export)


class Scale:
    """Colour/height range shared by the camera views, the bar and both 3D views."""

    def __init__(self, low, high, unit, auto, cfg):
        self.low, self.high, self.unit, self.auto, self.cfg = low, high, unit, auto, cfg
        self.primed = False

    @property
    def span(self):
        return max(self.high - self.low, 1e-6)

    @property
    def label(self):
        return "°C" if self.unit == "C" else self.unit

    def update(self, values):
        """Follow the scene smoothly; rise fast so hot things never clip."""
        if not self.auto:
            return
        cfg = self.cfg
        lo, hi = np.percentile(values[::2, ::2], (cfg.scale_low_pct, cfg.scale_high_pct))
        hi = max(hi, lo + cfg.min_span)
        if not self.primed:
            self.low, self.high, self.primed = float(lo), float(hi), True
            return
        self.low += cfg.scale_smoothing * (lo - self.low)
        self.high += (0.5 if hi > self.high else cfg.scale_smoothing) * (hi - self.high)


def decode_temps(frame):
    """Return a 192x256 array in °C from a raw TC001 frame."""
    data = frame.reshape(FRAME_SHAPE)[FRAME_SHAPE[0] // 2:]
    raw = data[..., 0].astype(np.uint16) + data[..., 1].astype(np.uint16) * 256
    return (raw / 64.0 - 273.15).astype(np.float32)


def decode_gray(frame, crop):
    """Return the cropped brightness image (0 to 255) from a normal colour frame."""
    return cv.cvtColor(frame[:, crop:], cv.COLOR_BGR2GRAY).astype(np.float32)


def open_raw(index):
    """Return a capture that delivers raw TC001 frames, or None if it doesn't."""
    for name, backend in (("DSHOW", cv.CAP_DSHOW), ("MSMF", cv.CAP_MSMF), ("default", cv.CAP_ANY)):
        capture = cv.VideoCapture(index, backend)
        if not capture.isOpened():
            capture.release()
            continue
        capture.set(cv.CAP_PROP_CONVERT_RGB, 0)
        ok, frame = capture.read()
        print(f"[{name}] read ok={ok}, shape={getattr(frame, 'shape', None)}, "
              f"size={getattr(frame, 'size', None)} (need {FRAME_SIZE})")
        if ok and frame.size == FRAME_SIZE:
            return capture
        capture.release()
    return None


class Camera:
    """Reads frames on a background thread so the display never works from stale, queued frames."""

    def __init__(self, cfg):
        self.crop = cfg.left_crop
        self.capture = open_raw(cfg.camera_index)
        self.raw = self.capture is not None
        if not self.raw:
            self.capture = cv.VideoCapture(cfg.camera_index)
        if not self.capture.isOpened():
            raise RuntimeError("Could not open camera. Try another camera_index.")
        if self.raw:
            self.scale = Scale(cfg.temp_min, cfg.temp_max, "C", cfg.auto_scale, cfg)
        else:
            self.scale = Scale(0, 255, "signal", False, cfg)

        self._frame = None
        self._lock = threading.Lock()
        self._ready = threading.Event()
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._grab, daemon=True)
        self._thread.start()

    def _grab(self):
        while not self._stop.is_set():
            ok, frame = self.capture.read()
            with self._lock:
                self._frame = frame if ok else None
            self._ready.set()
            if not ok:
                break

    def read(self):
        """Return the newest temperatures in °C (raw mode) or brightness, or None."""
        if not self._ready.wait(2.0):
            return None
        self._ready.clear()
        with self._lock:
            frame = self._frame
        if frame is None:
            return None
        return decode_temps(frame) if self.raw else decode_gray(frame, self.crop)

    def release(self):
        self._stop.set()
        self._thread.join(timeout=2.0)
        self.capture.release()


class SimulatedCamera:
    """Drop-in stand-in for Camera: same attributes and methods, synthetic 192x256 °C frames.

    Scene: a cool room with a warm right-hand wall and slow drifting texture, a person
    (body + head) walking side to side, a pulsing hot spot (like a soldering iron or mug),
    and a warm hand that sweeps across now and then. Plus a little sensor noise.
    """

    H, W = 192, 256

    def __init__(self, cfg):
        self.raw = True
        self.crop = cfg.left_crop
        self.scale = Scale(cfg.temp_min, cfg.temp_max, "C", cfg.auto_scale, cfg)
        self.period = 1.0 / cfg.sim_fps
        self.rng = np.random.default_rng(1)
        self.gx = np.arange(self.W, dtype=np.float32)
        self.gy = np.arange(self.H, dtype=np.float32)

        # Static, smooth room texture (about +/-1 degree) and a warm wall on the right.
        coarse = self.rng.normal(size=(12, 16)).astype(np.float32)
        texture = cv.resize(coarse, (self.W, self.H), interpolation=cv.INTER_CUBIC)
        wall = 3.0 * np.clip((self.gx - 150) / 106, 0, 1) ** 2
        self.room = 21.0 + 1.0 * texture + wall[None, :]

        self.t0 = time.monotonic()
        self.next_frame = self.t0

    def _blob(self, cx, cy, sx, sy, amp):
        bx = np.exp(-((self.gx - cx) ** 2) / (2 * sx * sx))
        by = np.exp(-((self.gy - cy) ** 2) / (2 * sy * sy))
        return amp * by[:, None] * bx[None, :]

    def read(self):
        """Block until the next frame is due, then return it (°C)."""
        now = time.monotonic()
        if now < self.next_frame:
            time.sleep(self.next_frame - now)
        self.next_frame = max(self.next_frame + self.period, time.monotonic() - self.period)
        t = time.monotonic() - self.t0

        frame = self.room.copy()

        # Person walking back and forth, with a bit of bobbing.
        px = self.W / 2 + 70 * np.sin(0.35 * t)
        py = self.H * 0.62 + 6 * np.sin(1.1 * t)
        frame += self._blob(px, py, 22, 48, 11.0)                 # torso
        frame += self._blob(px, py - 58, 10, 12, 13.0)            # head

        # Pulsing hot spot, just beyond where the person turns around.
        frame += self._blob(238, 150, 5, 5, 24 + 8 * np.sin(1.7 * t))

        # A warm hand sweeping across the top every ~9 s.
        phase = (t % 9.0) / 9.0
        frame += self._blob(-20 + phase * (self.W + 40), 50, 8, 8, 14 * np.sin(np.pi * phase))

        frame += self.rng.normal(0, 0.08, frame.shape)
        return frame.astype(np.float32)

    def release(self):
        pass


def make_camera(cfg):
    return SimulatedCamera(cfg) if cfg.simulate else Camera(cfg)


# ----------------------------------------------------------------------------------------------
# Hotspot detection and tracking
# ----------------------------------------------------------------------------------------------

@dataclass
class Detection:
    box: tuple          # x, y, w, h in image pixels
    center: tuple       # heat-weighted centre (x, y)
    peak: float         # hottest reading inside the hotspot
    peak_xy: tuple      # where that reading is
    mean: float         # average reading over the hotspot
    area: int           # pixels
    mask: np.ndarray    # bool image, True inside the hotspot


def segment_peaks(img, threshold, prominence):
    """Label the pixels above threshold, one label per peak that clearly stands on its own.

    Pixels are visited hottest first. Each one joins the region of a neighbour it touches.
    When two regions meet, the cooler one is folded into the hotter one unless its peak rises
    at least `prominence` above the meeting point. So a head on a body stays one hotspot,
    while a person standing next to a hot mug gives two.
    """
    h, w = img.shape
    pw = w + 2
    padded = np.full((h + 2, pw), -np.inf, np.float32)
    padded[1:-1, 1:-1] = img
    flat = padded.ravel()
    idx = np.flatnonzero(flat > threshold)
    if idx.size == 0:
        return np.zeros((h, w), np.int32)

    order = idx[np.argsort(-flat[idx], kind="stable")]
    offsets = (-pw - 1, -pw, -pw + 1, -1, 1, pw - 1, pw, pw + 1)
    owner, parent, peak = {}, [], []

    def find(c):
        while parent[c] != c:
            parent[c] = parent[parent[c]]
            c = parent[c]
        return c

    for p, v in zip(order.tolist(), flat[order].tolist()):
        roots = []
        for o in offsets:
            c = owner.get(p + o)
            if c is not None:
                r = find(c)
                if r not in roots:
                    roots.append(r)
        if not roots:
            owner[p] = len(parent)
            parent.append(len(parent))
            peak.append(v)
            continue
        if len(roots) > 1:
            roots.sort(key=peak.__getitem__, reverse=True)
            for r in roots[1:]:
                if peak[r] - v < prominence:
                    parent[r] = roots[0]
        owner[p] = roots[0]

    names = {}
    pixels = np.fromiter(owner.keys(), np.int64, len(owner))
    labels = np.fromiter((names.setdefault(find(c), len(names) + 1) for c in owner.values()),
                         np.int32, len(owner))
    out = np.zeros(flat.size, np.int32)
    out[pixels] = labels
    return out.reshape(h + 2, pw)[1:-1, 1:-1].copy()


class HotspotDetector:
    """Finds the regions clearly hotter than the scene and measures each one."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.ambient = None       # scene median, smoothed
        self.threshold = None     # current "hotspot" level

    def detect(self, values, scale):
        cfg = self.cfg
        blurred = cv.GaussianBlur(values, (0, 0), 1.2)
        target = float(np.median(blurred[::2, ::2]))
        self.ambient = target if self.ambient is None else self.ambient + 0.1 * (target - self.ambient)
        self.threshold = self.ambient + max(cfg.hot_delta, cfg.hot_fraction * scale.span)
        prominence = max(cfg.prominence, 0.08 * scale.span)

        h, w = values.shape
        f = max(1, int(cfg.detect_step))
        small = cv.resize(blurred, (w // f, h // f), interpolation=cv.INTER_AREA) if f > 1 else blurred
        small_labels = segment_peaks(small, self.threshold, prominence)
        up = np.repeat(np.repeat(small_labels, f, axis=0), f, axis=1)
        labels = np.zeros((h, w), np.int32)
        labels[:up.shape[0], :up.shape[1]] = up
        labels[blurred <= self.threshold] = 0

        found = []
        for label in range(1, int(small_labels.max()) + 1):
            mask = labels == label
            ys, xs = np.nonzero(mask)
            if xs.size < cfg.min_area:
                continue
            weight = np.clip(blurred[ys, xs] - self.ambient, 1e-3, None)
            if blurred[ys, xs].max() < self.threshold + cfg.min_rise:
                continue                      # barely warm: texture or the edge of something
            readings = values[ys, xs]
            k = int(np.argmax(readings))
            x0, y0 = int(xs.min()), int(ys.min())
            found.append(Detection(
                box=(x0, y0, int(xs.max()) - x0 + 1, int(ys.max()) - y0 + 1),
                center=(float((xs * weight).sum() / weight.sum()),
                        float((ys * weight).sum() / weight.sum())),
                peak=float(readings[k]),
                peak_xy=(int(xs[k]), int(ys[k])),
                mean=float(readings.mean()),
                area=int(xs.size),
                mask=mask,
            ))
        return found


class Track:
    """One hotspot followed over time, with everything recorded about it."""

    def __init__(self, det, now, cfg):
        self.id = None                    # given once the hotspot is confirmed
        self.det = det
        self.center = self.start = self.pos = det.center   # smoothed centre / start / raw centre
        self.vel = (0.0, 0.0)                              # px per second, smoothed
        self.first_seen = self.last_seen = now
        self.hits, self.missed = 0, 0
        self.distance = 0.0
        self.high, self.low, self.total = -np.inf, np.inf, 0.0
        self.log = deque(maxlen=cfg.log_length)   # (t, peak, mean, cx, cy, bx, by, bw, bh)
        self.see(det, now)

    def see(self, det, now):
        if self.hits:
            dt = now - self.last_seen
            if dt > 0:
                vx = (det.center[0] - self.pos[0]) / dt
                vy = (det.center[1] - self.pos[1]) / dt
                self.vel = (self.vel[0] + 0.4 * (vx - self.vel[0]), self.vel[1] + 0.4 * (vy - self.vel[1]))
            self.pos = det.center
            cx, cy = self.center
            nx, ny = cx + 0.5 * (det.center[0] - cx), cy + 0.5 * (det.center[1] - cy)   # smoothed
            self.distance += float(np.hypot(nx - cx, ny - cy))
            self.center = (nx, ny)
        self.det, self.last_seen = det, now
        self.hits += 1
        self.missed = 0
        self.high, self.low = max(self.high, det.peak), min(self.low, det.peak)
        self.total += det.peak
        self.log.append((now, det.peak, det.mean, *self.center, *det.box))

    def predict(self, now):
        """Where the hotspot should be now, from its last position and velocity."""
        dt = min(now - self.last_seen, 1.0)
        return self.pos[0] + self.vel[0] * dt, self.pos[1] + self.vel[1] * dt

    @property
    def average(self):
        return self.total / max(self.hits, 1)

    def speed(self, window=1.0):
        """Pixels per second over the last `window` seconds of sightings."""
        if len(self.log) < 2:
            return 0.0
        last = self.log[-1]
        old = last
        for entry in reversed(self.log):
            old = entry
            if entry[0] <= last[0] - window:
                break
        dt = last[0] - old[0]
        return float(np.hypot(last[3] - old[3], last[4] - old[4]) / dt) if dt > 0 else 0.0

    def recent(self, since):
        """Log entries newer than `since`, oldest first."""
        return list(takewhile(lambda e: e[0] >= since, reversed(self.log)))[::-1]


class Tracker:
    """Keeps hotspot IDs stable from frame to frame by matching nearest centres."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.tracks = []
        self.next_id = 1

    def update(self, detections, now, keep=None):
        cfg = self.cfg
        pairs = []
        for i, t in enumerate(self.tracks):
            px, py = t.predict(now)
            reach = cfg.match_distance + cfg.max_speed * min(now - t.last_seen, 0.75)
            for j, d in enumerate(detections):
                dist = float(np.hypot(px - d.center[0], py - d.center[1]))
                if dist <= reach:
                    pairs.append((dist, i, j))
        pairs.sort()
        used_t, used_d = set(), set()
        for _, i, j in pairs:
            if i in used_t or j in used_d:
                continue
            used_t.add(i)
            used_d.add(j)
            self.tracks[i].see(detections[j], now)

        survivors = []
        for i, t in enumerate(self.tracks):
            if i not in used_t:
                t.missed += 1
                limit = cfg.forget_frames if t.id is not None else 1
                if t.missed > limit and (keep is None or t.id != keep):
                    continue
            if t.id is None and t.hits >= cfg.confirm_frames:
                t.id = self.next_id
                self.next_id += 1
            survivors.append(t)
        survivors += [Track(d, now, cfg) for j, d in enumerate(detections) if j not in used_d]
        self.tracks = survivors

    def get(self, tid):
        if tid is None:
            return None
        return next((t for t in self.tracks if t.id == tid), None)

    def listed(self):
        """Confirmed hotspots, by ID."""
        return sorted((t for t in self.tracks if t.id is not None), key=lambda t: t.id)


# ----------------------------------------------------------------------------------------------
# 3D views
# ----------------------------------------------------------------------------------------------

class History:
    """Rolling rows: hottest value in each block of image columns, averaged over each time step."""

    def __init__(self, width, cfg):
        self.step = max(1, width // cfg.columns)
        self.n = width // self.step
        self.x = np.arange(self.n) * self.step + self.step / 2
        self.rows = deque(maxlen=cfg.rows)
        self.reset()

    def reset(self):
        self.rows.clear()
        self.total = np.zeros(self.n, np.float32)
        self.count = 0

    def add_frame(self, values):
        blurred = cv.GaussianBlur(values, (0, 0), 1.0)        # suppress single hot pixels
        h = blurred.shape[0]
        blocks = blurred[:, :self.n * self.step].reshape(h, self.n, self.step)
        self.total += blocks.max(axis=(0, 2))                  # hottest pixel per column block
        self.count += 1

    def commit(self):
        if self.count == 0:
            return
        row = self.total / self.count
        if not self.rows:
            self.rows.extend([row] * self.rows.maxlen)
        else:
            self.rows.append(row)
        self.total = np.zeros(self.n, np.float32)
        self.count = 0

    def as_array(self):
        return np.array(self.rows, np.float32)


BG, PANEL, EDGE = "#0c0e13", "#151821", "#2b303b"
TEXT, MUTED = "#e3e6ec", "#8b92a3"
SPOT_COLORS = ("#38d1ff", "#7dff6a", "#ff6bd6", "#f4f4f4", "#b69bff", "#00f5c4", "#6ba4ff", "#d4ff3a")


def spot_color(tid):
    return SPOT_COLORS[(tid - 1) % len(SPOT_COLORS)]


def style_dark_axes(ax, xlim, ylim, zlabel):
    ax.set_facecolor(BG)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_pane_color((0, 0, 0, 0))
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Image X", color="0.7")
    ax.set_ylabel("Time", color="0.7")
    ax.set_zlabel(zlabel, color="0.7")
    ax.tick_params(colors="0.6")
    ax.view_init(45, -70)


class SurfacePanel:
    """Surface of the hottest value per column over time, in real units."""

    def __init__(self, ax, history, cfg, scale):
        self.ax, self.scale, self.surf = ax, scale, None
        self.x, self.y = np.meshgrid(history.x, np.arange(cfg.rows))
        style_dark_axes(ax, (0, history.x.max()), (0, cfg.rows - 1), f"Signal ({scale.label})")

    def draw(self, grid):
        s = self.scale
        if self.surf is not None:
            self.surf.remove()
        self.surf = self.ax.plot_surface(
            self.x, self.y, np.clip(grid, s.low, s.high),
            cmap="inferno", vmin=s.low, vmax=s.high,
            linewidth=0, antialiased=False, shade=False,
            rcount=grid.shape[0], ccount=grid.shape[1],
        )
        self.ax.set_zlim(s.low, s.high)


class WispPanel:
    """Particles for everything hotter than the scene's normal level."""

    PARTICLES = 2

    def __init__(self, ax, history, cfg, scale):
        self.ax, self.cfg, self.scale, self.history = ax, cfg, scale, history
        self.x, self.y = np.meshgrid(history.x, np.arange(cfg.rows))
        rng = np.random.default_rng(7)
        self.jitter = rng.uniform(-0.5, 0.5, (self.PARTICLES, 3, self.x.size))   # fixed, so no flicker
        self.fade = np.linspace(0.15, 1.0, cfg.rows)[:, None]
        self.ambient = None
        self.top = None
        style_dark_axes(ax, (0, history.x.max()), (0, cfg.rows - 1), f"{scale.label} above normal")
        self.cloud = ax.scatter([0], [0], [0], s=1, c=[(0, 0, 0, 0)], depthshade=False, linewidths=0)

    def draw(self, grid):
        s, cfg = self.scale, self.cfg
        target = float(np.percentile(grid, cfg.ambient_pct))
        self.ambient = target if self.ambient is None else self.ambient + 0.05 * (target - self.ambient)

        contrast = max(cfg.contrast * s.span, 1e-6)
        excess = np.clip(grid - self.ambient, 0, None)
        strength = np.clip(excess / contrast, 0, 1) * self.fade
        cells = np.flatnonzero(strength.ravel() > cfg.min_strength)

        target_top = max(4 * contrast, float(excess.max()) * 1.1)
        self.top = target_top if self.top is None else self.top + 0.2 * (target_top - self.top)
        self.ax.set_zlim(0, self.top)

        if not cells.size:
            self.cloud._offsets3d = ([0.0], [0.0], [0.0])
            self.cloud.set_facecolor([(0, 0, 0, 0)])
            self.cloud.set_sizes([1])
            return

        heat = excess.ravel()[cells]
        power = strength.ravel()[cells]
        jit = self.jitter[:, :, cells]                                        # (P, 3, n)
        px = (self.x.ravel()[cells] + jit[:, 0] * self.history.step).ravel()
        py = (self.y.ravel()[cells] + jit[:, 1]).ravel()
        pz = np.clip(heat + jit[:, 2] * contrast * 0.15, 0, None).ravel()
        power = np.tile(power, self.PARTICLES)
        temp = self.ambient + np.tile(heat, self.PARTICLES)                    # absolute value -> same colour bar

        colors = plt.get_cmap("inferno")(np.clip((temp - s.low) / s.span, 0, 1))
        colors[:, 3] = np.clip(power * 1.2, 0, 1)
        self.cloud._offsets3d = (px, py, pz)
        self.cloud.set_facecolor(colors)
        self.cloud.set_sizes(4 + 34 * power)


# ----------------------------------------------------------------------------------------------
# The one window
# ----------------------------------------------------------------------------------------------

def style_panel(ax):
    ax.set_facecolor(PANEL)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    for spine in ax.spines.values():
        spine.set_color(EDGE)


class ThermalApp:
    """Live map with every hotspot boxed, a tracked-only view, both 3D views and data panels.

    Pick a hotspot by clicking its box on the live map or its row in the list. The live map
    keeps showing everything; the tracked view, the 3D views, the data and the chart then
    follow only the picked hotspot.
    """

    ROWS, ROW_TOP, ROW_GAP = 8, 0.80, 0.095

    def __init__(self, cfg, scale, history, tracker, detector, shape, simulated):
        plt.ion()
        self.cfg, self.scale, self.history = cfg, scale, history
        self.tracker, self.detector = tracker, detector
        self.selected = None
        self.closed = False
        self.message, self.message_until = "", 0.0
        self.row_ids, self.drawn, self.buttons = [], [], []
        self.frames, self.fps, self.fps_t0 = 0, 0.0, time.monotonic()
        self.soft_key, self.soft = None, None
        m = max(0, int(cfg.focus_margin))
        self.kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (2 * m + 1, 2 * m + 1))
        h, w = shape
        self.width = w

        fig = self.fig = plt.figure("Thermal hotspot tracker", figsize=cfg.figure_size, facecolor=BG)
        manager = fig.canvas.manager
        if getattr(manager, "key_press_handler_id", None) is not None:
            fig.canvas.mpl_disconnect(manager.key_press_handler_id)   # our own keys, not matplotlib's

        fig.text(0.012, 0.972, "Thermal hotspot tracker" + ("   ·   SIMULATION" if simulated else ""),
                 color=TEXT, fontsize=14, fontweight="bold", va="center")
        fig.text(0.012, 0.944, "Click a box (or a list row) to track one hotspot   ·   N / P  next / previous"
                 "   ·   A  show all   ·   1-8  pick from list   ·   S  save CSV   ·   Q  quit",
                 color=MUTED, fontsize=9, va="center")

        # 2D views: live map (all hotspots) and tracked-only view
        self.ax_live = fig.add_axes([0.012, 0.50, 0.325, 0.41])
        self.ax_focus = fig.add_axes([0.35, 0.50, 0.325, 0.41])
        blank = np.zeros(shape, np.float32)
        self.live_img = self._image_axes(self.ax_live, blank, w, h)
        self.focus_img = self._image_axes(self.ax_focus, blank, w, h)
        self.live_boxes, self.focus_boxes = [], []
        self.live_peak, = self.ax_live.plot([], [], "+", ms=12, mew=1.6, color="white")
        self.focus_peak, = self.ax_focus.plot([], [], "+", ms=12, mew=1.6, color="white")
        self.focus_note = self.ax_focus.text(0.5, 0.5, "", transform=self.ax_focus.transAxes, ha="center",
                                             va="center", color=TEXT, fontsize=11,
                                             bbox=dict(boxstyle="round,pad=0.5", fc=BG, ec=EDGE, alpha=0.85))
        self.focus_note.set_visible(False)
        self.live_title = self.ax_live.text(0, 1.015, "", transform=self.ax_live.transAxes, color=TEXT,
                                            fontsize=10, va="bottom")
        self.focus_title = self.ax_focus.text(0, 1.015, "", transform=self.ax_focus.transAxes, color=TEXT,
                                              fontsize=10, va="bottom")

        cbar = fig.colorbar(self.live_img, cax=fig.add_axes([0.684, 0.52, 0.008, 0.37]))
        cbar.ax.set_title(scale.label, color=MUTED, fontsize=9)
        cbar.ax.tick_params(colors=MUTED, labelsize=8)
        cbar.outline.set_edgecolor(EDGE)

        # 3D views
        left = fig.add_axes([-0.01, 0.0, 0.37, 0.48], projection="3d")
        right = fig.add_axes([0.33, 0.0, 0.37, 0.48], projection="3d")
        self.panels = [SurfacePanel(left, history, cfg, scale), WispPanel(right, history, cfg, scale)]

        # hotspot list
        self.ax_list = fig.add_axes([0.715, 0.70, 0.272, 0.225])
        style_panel(self.ax_list)
        self.list_title = self.ax_list.text(0.04, 0.93, "", color=MUTED, fontsize=9, fontweight="bold", va="top")
        self.row_bg = Rectangle((0.015, 0), 0.97, self.ROW_GAP * 0.92, fc="#22314a", ec="none", visible=False)
        self.ax_list.add_patch(self.row_bg)
        self.row_swatch, self.row_text = [], []
        for i in range(self.ROWS):
            y = self.ROW_TOP - i * self.ROW_GAP
            self.row_swatch.append(self.ax_list.text(0.04, y, "■", fontsize=11, va="center"))
            self.row_text.append(self.ax_list.text(0.10, y, "", family="monospace", fontsize=9.5,
                                                   color=TEXT, va="center"))
        self.list_note = self.ax_list.text(0.5, 0.45, "", ha="center", va="center", color=MUTED, fontsize=9.5)

        # data panel
        self.ax_info = fig.add_axes([0.715, 0.37, 0.272, 0.315])
        style_panel(self.ax_info)
        self.info_title = self.ax_info.text(0.04, 0.95, "", color=MUTED, fontsize=9, fontweight="bold", va="top")
        self.info_keys = self.ax_info.text(0.04, 0.865, "", color=MUTED, fontsize=9.5, va="top", linespacing=1.5)
        self.info_vals = self.ax_info.text(0.42, 0.865, "", color=TEXT, fontsize=9.5, va="top", linespacing=1.5)

        # temperature chart
        ax = self.ax_chart = fig.add_axes([0.752, 0.145, 0.233, 0.18])
        ax.set_facecolor(PANEL)
        for spine in ax.spines.values():
            spine.set_color(EDGE)
        ax.tick_params(colors=MUTED, labelsize=8)
        ax.grid(color=EDGE, linewidth=0.6)
        ax.set_xlabel("seconds ago", color=MUTED, fontsize=8)
        ax.set_ylabel(scale.label, color=MUTED, fontsize=8)
        ax.set_xlim(-cfg.chart_seconds, 0)
        self.chart_lines = []
        self.mean_line, = ax.plot([], [], "--", lw=1, color=MUTED)
        self.peak_line = ax.axhline(0, ls=":", lw=1, color=MUTED, visible=False)

        # buttons and status line
        self.message_text = fig.text(0.715, 0.093, "", color="#ffd166", fontsize=9, va="center")
        for i, (label, action) in enumerate((("◀ Prev", lambda: self.cycle(-1)),
                                             ("Next ▶", lambda: self.cycle(1)),
                                             ("Show all", lambda: self.select(None)),
                                             ("Save CSV", self.save_csv))):
            self._button([0.715 + i * 0.0703, 0.025, 0.062, 0.05], label, action)

        fig.canvas.mpl_connect("button_press_event", self._on_click)
        fig.canvas.mpl_connect("key_press_event", self._on_key)
        fig.canvas.mpl_connect("close_event", self._on_close)

        # Fast parts are "animated": drawn on top of a saved background (blitting), so the live
        # map stays smooth while the slow 3D views only redraw every full_update_seconds.
        self.blit = bool(getattr(fig.canvas, "supports_blit", False))
        self.background, self.last_full = None, -np.inf
        self.fast = [self.live_img, self.focus_img, self.live_peak, self.focus_peak, self.focus_note,
                     self.live_title, self.focus_title, self.message_text]
        for artist in self.fast:
            artist.set_animated(self.blit)
        fig.canvas.mpl_connect("draw_event", self._on_draw)
        try:
            fig.show()
        except Exception:
            pass

    # ---- building blocks ----

    def _image_axes(self, ax, blank, w, h):
        image = ax.imshow(blank, cmap="inferno", vmin=self.scale.low, vmax=self.scale.high,
                          interpolation="bilinear")
        ax.set_xlim(-0.5, w - 0.5)
        ax.set_ylim(h - 0.5, -0.5)
        ax.set_autoscale_on(False)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_color(EDGE)
        return image

    def _button(self, rect, label, action):
        ax = self.fig.add_axes(rect)
        button = Button(ax, label, color=PANEL, hovercolor="#232838")
        button.label.set_color(TEXT)
        button.label.set_fontsize(9)
        for spine in ax.spines.values():
            spine.set_color(EDGE)
        button.on_clicked(lambda _event: action())
        self.buttons.append(button)          # widgets stop working if nothing holds them

    def _box(self, ax, pool, i):
        while len(pool) <= i:
            rect = Rectangle((0, 0), 1, 1, fill=False, lw=1.2)
            ax.add_patch(rect)
            label = ax.text(0, 0, "", fontsize=8, fontweight="bold", clip_on=True,
                            bbox=dict(boxstyle="square,pad=0.15", fc="black", ec="none", alpha=0.6))
            for artist in (rect, label):
                artist.set_animated(self.blit)
            pool.append((rect, label))
        return pool[i]

    def _place_box(self, rect, label, t, width, style, alpha):
        x, y, w, h = t.det.box
        color = spot_color(t.id)
        rect.set_bounds(x - 0.5, y - 0.5, w, h)
        rect.set_edgecolor(color)
        rect.set_linewidth(width)
        rect.set_linestyle(style)
        rect.set_alpha(alpha)
        rect.set_visible(True)
        above = y > 14
        right = x > self.width - 60
        label.set_position((x + w - 0.5 if right else x - 0.5, y - 1.5 if above else y + h + 0.5))
        label.set_va("bottom" if above else "top")
        label.set_ha("right" if right else "left")
        label.set_text(f"#{t.id}  {t.det.peak:.1f} {self.scale.label}")
        label.set_color(color)
        label.set_alpha(alpha)
        label.set_visible(True)

    @staticmethod
    def _hide(pool, start=0):
        for rect, label in pool[start:]:
            rect.set_visible(False)
            label.set_visible(False)

    # ---- selection ----

    def select(self, tid):
        if tid == self.selected:
            return
        self.selected = tid
        self.history.reset()               # 3D views restart with only the picked hotspot
        self.soft_key = None
        self.last_full = -np.inf           # redraw the 3D views and chart right away
        self.flash(f"Tracking hotspot #{tid} only" if tid is not None else "Showing all hotspots")

    def cycle(self, step):
        ids = [t.id for t in self.tracker.listed()]
        if not ids:
            self.flash("No hotspots to pick")
            return
        if self.selected in ids:
            self.select(ids[(ids.index(self.selected) + step) % len(ids)])
        else:
            self.select(ids[0] if step > 0 else ids[-1])

    def flash(self, text, seconds=3.0):
        self.message, self.message_until = text, time.monotonic() + seconds

    def save_csv(self):
        t = self.tracker.get(self.selected)
        if t is None:
            self.flash("Pick a hotspot first, then save its data")
            return
        name = f"hotspot_{t.id}_{time.strftime('%Y%m%d_%H%M%S')}.csv"
        data = np.array(t.log, np.float64)
        data[:, 0] -= t.first_seen
        np.savetxt(name, data, delimiter=",", fmt="%.3f", comments="",
                   header="time_s,temp_max,temp_mean,center_x,center_y,box_x,box_y,box_w,box_h")
        print(f"Saved {len(data)} samples of hotspot #{t.id} to {name}")
        self.flash(f"Saved {name}")

    # ---- events ----

    def _on_click(self, event):
        if event.button != 1 or event.xdata is None:
            return
        toolbar = getattr(self.fig.canvas, "toolbar", None)
        if toolbar is not None and getattr(toolbar, "mode", ""):
            return                         # zoom/pan tool is active
        if event.inaxes is self.ax_live:
            x, y, pad = event.xdata, event.ydata, 3
            hits = [t for t in self.drawn
                    if t.det.box[0] - pad <= x <= t.det.box[0] + t.det.box[2] + pad
                    and t.det.box[1] - pad <= y <= t.det.box[1] + t.det.box[3] + pad]
            if hits:
                self.select(min(hits, key=lambda t: t.det.area).id)
            else:
                self.flash("No hotspot there - click inside a box")
        elif event.inaxes is self.ax_list:
            row = int(round((self.ROW_TOP - event.ydata) / self.ROW_GAP))
            if 0 <= row < len(self.row_ids):
                self.select(self.row_ids[row])

    def _on_key(self, event):
        key = (event.key or "").lower()
        if key in ("q", "escape"):
            self.closed = True
        elif key in ("n", "right", "down"):
            self.cycle(1)
        elif key in ("p", "left", "up"):
            self.cycle(-1)
        elif key in ("a", "0"):
            self.select(None)
        elif key == "s":
            self.save_csv()
        elif key.isdigit() and int(key) - 1 < len(self.row_ids):
            self.select(self.row_ids[int(key) - 1])

    def _on_close(self, _event):
        self.closed = True

    def _on_draw(self, _event):
        """After every full draw (ours, or a window resize): save the background, add the fast parts."""
        if self.blit:
            self.background = self.fig.canvas.copy_from_bbox(self.fig.bbox)
            self._draw_fast_artists()

    def _draw_fast_artists(self):
        boxes = [a for pair in self.live_boxes + self.focus_boxes for a in pair]
        for artist in self.fast[:2] + boxes + self.fast[2:]:
            if artist.get_visible():
                self.fig.draw_artist(artist)

    # ---- per frame ----

    def tick(self, now):
        self.frames += 1
        if now - self.fps_t0 >= 1.0:
            self.fps = self.frames / (now - self.fps_t0)
            self.frames, self.fps_t0 = 0, now

    def focus_values(self, values):
        """The frame with everything except the picked hotspot flattened to the scene median."""
        if self.selected is None:
            return values
        ambient = self.detector.ambient
        t = self.tracker.get(self.selected)
        if t is None:
            return np.full_like(values, ambient)
        key = (t.id, t.hits)
        if key != self.soft_key:
            grown = cv.dilate(t.det.mask.astype(np.uint8), self.kernel).astype(np.float32)
            self.soft = cv.GaussianBlur(grown, (0, 0), 1.5)
            self.soft_key = key
        return (ambient + (values - ambient) * self.soft).astype(np.float32)

    def draw(self, values, focused, now):
        """Refresh the whole window. Returns False once it has been closed."""
        s = self.scale
        for image, data in ((self.live_img, values), (self.focus_img, focused)):
            image.set_data(data)
            image.set_clim(s.low, s.high)
        listed = self.tracker.listed()
        picked = self.tracker.get(self.selected)
        self._draw_maps(listed, picked)
        self.message_text.set_text(self.message if time.monotonic() < self.message_until else "")

        canvas = self.fig.canvas
        full = (not self.blit or self.background is None
                or time.monotonic() - self.last_full >= self.cfg.full_update_seconds)
        if full:
            self._draw_list(listed, picked, now)
            self._draw_info(listed, picked, now)
            self._draw_chart(listed, picked, now)
            self._draw_3d()
            canvas.draw()                  # triggers _on_draw, which adds the fast parts
            self.last_full = time.monotonic()
        else:
            canvas.restore_region(self.background)
            self._draw_fast_artists()
            canvas.blit(self.fig.bbox)
        canvas.start_event_loop(0.001)
        return not self.closed and plt.fignum_exists(self.fig.number)

    def _draw_maps(self, listed, picked):
        shown = [t for t in listed if t.missed <= 2]
        self.drawn = shown
        for i, t in enumerate(shown):
            rect, label = self._box(self.ax_live, self.live_boxes, i)
            if picked is None:
                self._place_box(rect, label, t, 1.4, "-", 1.0)
            elif t is picked:
                self._place_box(rect, label, t, 2.6, "-", 1.0)
            else:
                self._place_box(rect, label, t, 1.0, "--", 0.45)
        self._hide(self.live_boxes, len(shown))
        n = len(shown)
        self.live_title.set_text(f"LIVE MAP  ·  {n} hotspot{'s' if n != 1 else ''}  ·  click a box to track it")

        if picked is not None:
            rect, label = self._box(self.ax_focus, self.focus_boxes, 0)
            self._place_box(rect, label, picked, 2.6, "-" if picked.missed == 0 else "--", 1.0)
            for marker in (self.live_peak, self.focus_peak):
                marker.set_data([picked.det.peak_xy[0]], [picked.det.peak_xy[1]])
            self.focus_note.set_visible(False)
            title = f"TRACKED ONLY  ·  hotspot #{picked.id}"
        else:
            self._hide(self.focus_boxes)
            for marker in (self.live_peak, self.focus_peak):
                marker.set_data([], [])
            gone = self.selected is not None
            self.focus_note.set_text(f"Hotspot #{self.selected} is gone\npick another one" if gone else
                                     "Nothing picked: showing every hotspot\nclick a box on the live map")
            self.focus_note.set_visible(gone or not shown)
            title = "TRACKED ONLY  ·  " + (f"#{self.selected} (gone)" if gone else "all hotspots")
        self.focus_title.set_text(title)

    def _draw_list(self, listed, picked, now):
        u = self.scale.label
        rows = listed[:self.ROWS]
        self.row_ids = [t.id for t in rows]
        more = len(listed) - len(rows)
        self.list_title.set_text(f"HOTSPOTS  ·  {len(listed)}" + (f"  (+{more} not shown)" if more else ""))
        for i, (swatch, text) in enumerate(zip(self.row_swatch, self.row_text)):
            if i >= len(rows):
                swatch.set_visible(False)
                text.set_visible(False)
                continue
            t = rows[i]
            status = "live" if t.missed == 0 else f"missing {now - t.last_seen:.0f}s"
            alpha = 0.45 if picked is not None and t is not picked else 1.0
            swatch.set_color(spot_color(t.id))
            text.set_text(f"#{t.id:<3d}{t.det.peak:6.1f} {u}   peak {t.high:5.1f}   {status}")
            for artist in (swatch, text):
                artist.set_alpha(alpha)
                artist.set_visible(True)
        if picked is not None and picked in rows:
            self.row_bg.set_y(self.ROW_TOP - rows.index(picked) * self.ROW_GAP - self.ROW_GAP * 0.46)
            self.row_bg.set_visible(True)
        else:
            self.row_bg.set_visible(False)
        self.list_note.set_text("" if rows else "No hotspots right now")

    def _draw_info(self, listed, picked, now):
        u, cfg = self.scale.label, self.cfg
        if picked is not None:
            t, d = picked, picked.det
            if t.missed == 0:
                status = "tracking"
            elif t.missed <= cfg.forget_frames:
                status = f"searching... {now - t.last_seen:.1f} s"
            else:
                status = f"lost, last seen {now - t.last_seen:.0f} s ago"
            cx, cy = t.center
            rows = [
                ("Hotspot", f"#{t.id}"),
                ("Status", status),
                ("Temp now", f"{d.peak:.1f} {u}" + ("" if t.missed == 0 else "  (last seen)")),
                ("Region mean", f"{d.mean:.1f} {u}"),
                ("Recorded peak", f"{t.high:.1f} {u}"),
                ("Recorded low", f"{t.low:.1f} {u}"),
                ("Recorded avg", f"{t.average:.1f} {u}"),
                ("Recorded for", f"{t.last_seen - t.first_seen:.1f} s  ({t.hits} samples)"),
                ("Position", f"x {cx:.0f}, y {cy:.0f} px"),
                ("Box size", f"{d.box[2]} x {d.box[3]} px  ({d.area} px)"),
                ("Distance moved", f"{t.distance:.0f} px"),
                ("From start", f"{np.hypot(cx - t.start[0], cy - t.start[1]):.0f} px"),
                ("Speed", f"{t.speed():.1f} px/s"),
            ]
            self.info_title.set_text(f"PICKED HOTSPOT  #{t.id}")
            self.info_title.set_color(spot_color(t.id))
        else:
            live = [t for t in listed if t.missed == 0]
            hottest = max(live, key=lambda t: t.det.peak, default=None)
            s = self.scale
            rows = [
                ("Tracking", "all hotspots" if self.selected is None else f"#{self.selected} (gone)"),
                ("Hotspots now", str(len(live))),
                ("Hottest", f"#{hottest.id}   {hottest.det.peak:.1f} {u}" if hottest else "-"),
                ("Scene median", f"{self.detector.ambient:.1f} {u}"),
                ("Hotspot above", f"{self.detector.threshold:.1f} {u}"),
                ("Colour scale", f"{s.low:.1f} to {s.high:.1f} {u}"),
                ("Frame rate", f"{self.fps:.1f} fps"),
                ("", ""),
                ("Tip", "click a box or a list row"),
                ("", "to track a single hotspot"),
            ]
            self.info_title.set_text("OVERVIEW")
            self.info_title.set_color(MUTED)
        self.info_keys.set_text("\n".join(k for k, _ in rows))
        self.info_vals.set_text("\n".join(v for _, v in rows))

    def _draw_chart(self, listed, picked, now):
        cfg, ax = self.cfg, self.ax_chart
        since = now - cfg.chart_seconds
        tracks = [picked] if picked is not None else [t for t in listed if t.missed <= cfg.forget_frames][:8]
        while len(self.chart_lines) < len(tracks):
            self.chart_lines.append(ax.plot([], [], lw=1.3)[0])
        lo, hi = np.inf, -np.inf
        for line, t in zip(self.chart_lines, tracks):
            data = np.array([(e[0], e[1], e[2]) for e in t.recent(since)]).reshape(-1, 3)
            line.set_data(data[:, 0] - now, data[:, 1])
            line.set_color(spot_color(t.id))
            line.set_linewidth(2.0 if picked is not None else 1.3)
            line.set_visible(True)
            if len(data):
                lo, hi = min(lo, data[:, 1:].min()), max(hi, data[:, 1].max())
            if t is picked:
                self.mean_line.set_data(data[:, 0] - now, data[:, 2])
        for line in self.chart_lines[len(tracks):]:
            line.set_visible(False)
        self.mean_line.set_visible(picked is not None)
        if picked is not None:
            self.peak_line.set_ydata([picked.high, picked.high])
            hi = max(hi, picked.high)
        self.peak_line.set_visible(picked is not None)
        if not np.isfinite(lo):
            lo, hi = self.scale.low, self.scale.high
        pad = max(0.5, 0.08 * (hi - lo))
        ax.set_ylim(lo - pad, hi + pad)
        ax.set_xlim(-cfg.chart_seconds, 0)
        title = (f"TEMPERATURE  #{picked.id}   (solid max, dashed mean, dotted recorded peak)"
                 if picked is not None else "TEMPERATURE  ·  all hotspots (max)")
        ax.set_title(title, color=MUTED, fontsize=8.5, loc="left")

    def _draw_3d(self):
        if not self.history.rows:
            return
        grid = cv.GaussianBlur(self.history.as_array(), (0, 0),
                               sigmaX=self.cfg.blur_x, sigmaY=self.cfg.blur_t)
        for panel in self.panels:
            panel.draw(grid)
        what = f"hotspot #{self.selected}" if self.selected is not None else "all hotspots"
        self.panels[0].ax.set_title(f"3D map  ·  {what}", color=TEXT, fontsize=10)
        self.panels[1].ax.set_title(f"Wisps  ·  {what}", color=TEXT, fontsize=10)


def run(cfg=Config()):
    if "inline" in matplotlib.get_backend().lower():
        raise RuntimeError("Inline backend closes the figure. Run %matplotlib qt, then restart the kernel.")

    camera = make_camera(cfg)
    if cfg.simulate:
        print("SIMULATION: synthetic thermal scene (set Config.simulate=False for the real camera).")
    else:
        print("Reading temperatures (°C)." if camera.raw else "Reading brightness only: this camera sends processed frames.")
    detector, tracker = HotspotDetector(cfg), Tracker(cfg)
    app = history = None
    last_update = 0.0

    try:
        while True:
            values = camera.read()
            if values is None:
                break
            now = time.monotonic()
            camera.scale.update(values)
            tracker.update(detector.detect(values, camera.scale), now,
                           keep=app.selected if app is not None else None)

            if app is None:
                history = History(values.shape[1], cfg)
                app = ThermalApp(cfg, camera.scale, history, tracker, detector, values.shape, cfg.simulate)

            app.tick(now)
            focused = app.focus_values(values)
            history.add_frame(focused)

            if now - last_update >= cfg.update_seconds:
                last_update = now
                history.commit()
                if not app.draw(values, focused, now):
                    break
    finally:
        camera.release()
        plt.close("all")


if __name__ == "__main__":
    run()

SIMULATION: synthetic thermal scene (set Config.simulate=False for the real camera).
